# 01 Data Overview

This notebook gives a first structured overview of the Spotify project data stored in DuckDB.  
The goal is to check which tables are available, how large they are, what columns they contain, and whether the data looks ready for merging and later recommendation modeling.

## 1. Setup

We first import the required packages, define the project root, and connect to the DuckDB database created in the previous setup step.

In [2]:
from pathlib import Path
import sys
import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from config.paths import DUCKDB_PATH

con = duckdb.connect(DUCKDB_PATH)

## 2. Available Tables

This step checks which tables are currently stored inside the DuckDB database.

In [3]:
tables = con.execute("SHOW TABLES").df()
tables

,name
0,albums
1,artists
2,audio_features
3,lyrics_features
4,tracks


## 3. Table Sizes

Here we check the number of rows and columns for each table.  
This helps us understand the scale of the data and identify which tables are large enough to be useful for analysis and recommendation modeling.

In [4]:
table_names = tables["name"].tolist()

for table in table_names:
    rows = con.execute(f"SELECT COUNT(*) AS n_rows FROM {table}").df()["n_rows"][0]
    cols = con.execute(f"DESCRIBE {table}").df().shape[0]
    print(f"{table}: {rows:,} rows, {cols} columns")

albums: 75,511 rows, 16 columns
artists: 56,129 rows, 9 columns
audio_features: 101,909 rows, 209 columns
lyrics_features: 94,954 rows, 8 columns
tracks: 101,939 rows, 32 columns


## 4. Column Overview

This section displays the schema of each table.  
We use this to understand available variables, data types, and possible join keys between the metadata and feature tables.

In [5]:
for table in table_names:
    print(f"\n--- {table} ---")
    display(con.execute(f"DESCRIBE {table}").df())


--- albums ---


,column_name,column_type,null,key,default,extra
0,column00,BIGINT,YES,None,None,None
1,album_type,VARCHAR,YES,None,None,None
2,artist_id,VARCHAR,YES,None,None,None
3,available_markets,VARCHAR,YES,None,None,None
4,external_urls,VARCHAR,YES,None,None,None
5,href,VARCHAR,YES,None,None,None
6,id,VARCHAR,YES,None,None,None
7,images,VARCHAR,YES,None,None,None
8,name,VARCHAR,YES,None,None,None
9,release_date,VARCHAR,YES,None,None,None



--- artists ---


,column_name,column_type,null,key,default,extra
0,column0,BIGINT,YES,None,None,None
1,artist_popularity,BIGINT,YES,None,None,None
2,followers,BIGINT,YES,None,None,None
3,genres,VARCHAR,YES,None,None,None
4,id,VARCHAR,YES,None,None,None
5,name,VARCHAR,YES,None,None,None
6,track_id,VARCHAR,YES,None,None,None
7,track_name_prev,VARCHAR,YES,None,None,None
8,type,VARCHAR,YES,None,None,None



--- audio_features ---


,column_name,column_type,null,key,default,extra
0,column000,BIGINT,YES,None,None,None
1,Chroma_1,DOUBLE,YES,None,None,None
2,Chroma_10,DOUBLE,YES,None,None,None
3,Chroma_11,DOUBLE,YES,None,None,None
4,Chroma_12,DOUBLE,YES,None,None,None
...,...,...,...,...,...,...
204,spectral_bandwith,DOUBLE,YES,None,None,None
205,spectral_centroid,DOUBLE,YES,None,None,None
206,spectral_rollOff_max,DOUBLE,YES,None,None,None
207,spectral_rollOff_min,DOUBLE,YES,None,None,None



--- lyrics_features ---


,column_name,column_type,null,key,default,extra
0,column0,BIGINT,YES,None,None,None
1,mean_syllables_word,DOUBLE,YES,None,None,None
2,mean_words_sentence,DOUBLE,YES,None,None,None
3,n_sentences,BIGINT,YES,None,None,None
4,n_words,BIGINT,YES,None,None,None
5,sentence_similarity,DOUBLE,YES,None,None,None
6,track_id,VARCHAR,YES,None,None,None
7,vocabulary_wealth,DOUBLE,YES,None,None,None



--- tracks ---


,column_name,column_type,null,key,default,extra
0,column00,BIGINT,YES,None,None,None
1,acousticness,DOUBLE,YES,None,None,None
2,album_id,VARCHAR,YES,None,None,None
3,analysis_url,VARCHAR,YES,None,None,None
4,artists_id,VARCHAR,YES,None,None,None
5,available_markets,VARCHAR,YES,None,None,None
6,country,VARCHAR,YES,None,None,None
7,danceability,DOUBLE,YES,None,None,None
8,disc_number,DOUBLE,YES,None,None,None
9,duration_ms,DOUBLE,YES,None,None,None


## 5. First Data Preview

Finally, we preview the first few rows of each table.  
This is useful for checking whether the columns are readable, whether the values look reasonable, and how the tables might connect to each other.

In [6]:
for table in table_names:
    print(f"\n--- {table} preview ---")
    display(con.execute(f"SELECT * FROM {table} LIMIT 5").df())


--- albums preview ---


,column00,album_type,artist_id,available_markets,external_urls,href,id,images,name,release_date,release_date_precision,total_tracks,track_id,track_name_prev,uri,type
0,0,single,3DiDSECUqqY1AuBP8qtaIa,"['AD', 'AE', 'AR', 'AT', 'AU', 'BE', 'BG', 'BH...",{'spotify': 'https://open.spotify.com/album/1g...,https://api.spotify.com/v1/albums/1gAM7M4rBwEb...,1gAM7M4rBwEbSPeAQR2nx1,"[{'height': 640, 'url': 'https://i.scdn.co/ima...",If I Ain't Got You EP,2019-02-08,day,6,2iejTMy9XZ8Gaae0aQ2yl0,track_32,spotify:album:1gAM7M4rBwEbSPeAQR2nx1,album
1,1,album,6s1pCNXcbdtQJlsnM1hRIA,"['AD', 'AE', 'AR', 'AT', 'AU', 'BE', 'BG', 'BH...",{'spotify': 'https://open.spotify.com/album/4K...,https://api.spotify.com/v1/albums/4KfJZV7WfolY...,4KfJZV7WfolYlxBzOTo66s,"[{'height': 640, 'url': 'https://i.scdn.co/ima...",Shostakovich Symphony No.5 - Four Romances on ...,2019-03-01,day,8,1WQfghEjszJJ4H8MAWrQ2C,track_11,spotify:album:4KfJZV7WfolYlxBzOTo66s,album
2,2,single,5YjfNaHq05WrwldRe1QSBc,"['AD', 'AE', 'AR', 'AT', 'AU', 'BE', 'BG', 'BH...",{'spotify': 'https://open.spotify.com/album/7n...,https://api.spotify.com/v1/albums/7nLYY7uAVUb5...,7nLYY7uAVUb57kpd7tZxnS,"[{'height': 640, 'url': 'https://i.scdn.co/ima...",Take My Bass,2019-03-14,day,1,3jJKj4QTK3v18ZSwpk7AcV,track_15,spotify:album:7nLYY7uAVUb57kpd7tZxnS,album
3,3,single,2G9Vc16JCpnZmK4uGH46Fa,"['AD', 'AE', 'AR', 'AT', 'AU', 'BE', 'BG', 'BH...",{'spotify': 'https://open.spotify.com/album/6p...,https://api.spotify.com/v1/albums/6p20Rt4x2Qn5...,6p20Rt4x2Qn5mUMRi1s6pj,"[{'height': 640, 'url': 'https://i.scdn.co/ima...",Hypnotizing (Are U),2016-11-16,day,1,1xGtDafUZbHyYC3Xarcbrj,track_46,spotify:album:6p20Rt4x2Qn5mUMRi1s6pj,album
4,4,single,2dwM9OcE4c3Ph1UBINSodx,"['AD', 'AE', 'AR', 'AT', 'AU', 'BE', 'BG', 'BH...",{'spotify': 'https://open.spotify.com/album/1X...,https://api.spotify.com/v1/albums/1XeoOqC1q7U2...,1XeoOqC1q7U2iyLEQJ64cu,"[{'height': 640, 'url': 'https://i.scdn.co/ima...",Sunshine,2018-07-20,day,1,0gWtsXvXOzAT6FtM3ur8in,track_10,spotify:album:1XeoOqC1q7U2iyLEQJ64cu,album



--- artists preview ---


,column0,artist_popularity,followers,genres,id,name,track_id,track_name_prev,type
0,0,44,23230,"['sertanejo', 'sertanejo pop', 'sertanejo trad...",4mGnpjhqgx4RUdsIJiURdo,Juliano Cezar,0wmDmAILuW9e2aRttkl4aC,track_9,artist
1,1,22,313,[],1dLnVku4VQUOLswwDFvRc9,The Grenadines,4wqwj0gA8qPZKLl5WVqXml,track_30,artist
2,2,26,1596,['danish pop rock'],6YVY310fjfUzKi8hiqR7iK,Gangway,1bFqWDbvHmZe2f4Nf9qaD8,track_38,artist
3,3,31,149,['uk alternative pop'],2VElyouiCfoYPDJluzwJwK,FADES,3MFSUBAidPzRBbIS7BDj1S,track_34,artist
4,4,21,11,['french baroque'],4agVy03qW8juSysCTUOuDI,Jean-Pierre Guignon,2r3q57FhxdsCyYr0kuDq4b,track_26,artist



--- audio_features preview ---


,column000,Chroma_1,Chroma_10,Chroma_11,Chroma_12,Chroma_2,Chroma_3,Chroma_4,Chroma_5,Chroma_6,...,Tonnetz_4,Tonnetz_5,Tonnetz_6,ZCR,entropy_energy,spectral_bandwith,spectral_centroid,spectral_rollOff_max,spectral_rollOff_min,track_id
0,0,0.438296,0.472769,0.427441,0.436688,0.467697,0.493862,0.512244,0.568658,0.560524,...,0.018434,-0.001759,-0.006392,0.067966,-89.113389,2564.247669,3558.400706,4508.506071,367.831109,19YEk4OVQZn3GfoxbpNrU6
1,1,0.596605,0.368288,0.285263,0.302211,0.905805,0.510909,0.221708,0.311248,0.491277,...,0.046941,0.005665,-0.026928,0.047308,-127.945239,2370.181495,1499.689590,3647.394611,230.165275,6zJms3MX11Qu1IKF44LoRW
2,2,0.505224,0.500420,0.506773,0.488258,0.498356,0.573582,0.690761,0.742858,0.686282,...,-0.006929,0.004968,0.008947,0.058463,-238.285176,2973.294736,1543.550034,5623.349330,187.290534,1WugzepXsLjnsM0K4UaWYc
3,3,0.525690,0.666469,0.579492,0.498920,0.598528,0.631578,0.501693,0.500468,0.587101,...,-0.027382,-0.009689,0.001402,0.080547,-148.785733,2716.749483,3017.248824,5799.931595,160.940693,1pSlTbCrUJ9rmwj5CNNrX4
4,4,0.632214,0.503698,0.496942,0.611532,0.634613,0.697265,0.557012,0.530836,0.444279,...,0.003728,-0.002780,-0.010120,0.084945,-176.618314,3096.692876,2118.686992,6560.018666,229.131948,5yruvWJs3mL00w4slpCVzN



--- lyrics_features preview ---


,column0,mean_syllables_word,mean_words_sentence,n_sentences,n_words,sentence_similarity,track_id,vocabulary_wealth
0,0,-1.00,-1.00,-1,-1,-1.000000,5KIfHjHI5NIsPHNt58qua0,-1.00
1,1,1.10,5.65,31,326,0.043011,13keyz9ikBe6ZpRasw7l4X,0.45
2,2,1.37,4.77,74,532,0.050352,1WugzepXsLjnsM0K4UaWYc,0.59
3,3,1.95,3.38,72,430,0.028560,2MO6oEAlMKcsfI8xP3yoy8,0.49
4,4,1.16,2.99,68,368,0.047849,1i4St7fmSUE9nB3R9n8fol,0.47



--- tracks preview ---


,column00,acousticness,album_id,analysis_url,artists_id,available_markets,country,danceability,disc_number,duration_ms,...,preview_url,speechiness,tempo,time_signature,track_href,track_name_prev,track_number,uri,valence,type
0,0,0.294,0D3QufeCudpQANOR7luqdr,https://api.spotify.com/v1/audio-analysis/5qlj...,['3mxJuHRn2ZWD5OofvJtDZY'],"['AD', 'AE', 'AR', 'AT', 'AU', 'BE', 'BG', 'BH...",BE,0.698,1.0,235584.0,...,https://p.scdn.co/mp3-preview/1b05a902da3a251d...,0.0262,115.018,4.0,https://api.spotify.com/v1/tracks/5qljLQuKnNJf...,track_14,1.0,spotify:track:5qljLQuKnNJf4F4vfxQB0V,0.6220,track
1,1,0.863,1bcqsH5UyTBzmh9YizdsBE,https://api.spotify.com/v1/audio-analysis/3VAX...,['4xWMewm6CYMstu0sPgd9jJ'],"['AD', 'AE', 'AR', 'AT', 'AU', 'BE', 'BG', 'BH...",BE,0.719,1.0,656960.0,...,https://p.scdn.co/mp3-preview/d8140736a6131cb5...,0.9220,115.075,3.0,https://api.spotify.com/v1/tracks/3VAX2MJdmdqA...,track_3,3.0,spotify:track:3VAX2MJdmdqARLSU5hPMpm,0.5890,track
2,2,0.750,4tKijjmxGClg4JOLAyo2qE,https://api.spotify.com/v1/audio-analysis/1L3Y...,['3hYaK5FF3YAglCj5HZgBnP'],['GB'],BE,0.466,1.0,492840.0,...,https://p.scdn.co/mp3-preview/c8af28fb15185b18...,0.9440,79.565,4.0,https://api.spotify.com/v1/tracks/1L3YAhsEMrGV...,track_4,4.0,spotify:track:1L3YAhsEMrGVvCgDXj2TYn,0.0850,track
3,3,0.763,6FeJF5r8roonnKraJxr4oB,https://api.spotify.com/v1/audio-analysis/6aCe...,['2KQsUB9DRBcJk17JWX1eXD'],"['AD', 'AE', 'AR', 'AT', 'AU', 'BE', 'BG', 'BH...",BE,0.719,1.0,316578.0,...,https://p.scdn.co/mp3-preview/7629b8e9f31f6e9b...,0.9380,112.822,3.0,https://api.spotify.com/v1/tracks/6aCe9zzoZmCo...,track_9,1.0,spotify:track:6aCe9zzoZmCojX7bbgKKtf,0.5330,track
4,4,0.770,4tKijjmxGClg4JOLAyo2qE,https://api.spotify.com/v1/audio-analysis/1Vo8...,['3hYaK5FF3YAglCj5HZgBnP'],['GB'],BE,0.460,1.0,558880.0,...,https://p.scdn.co/mp3-preview/32be593c0eb82868...,0.9430,81.260,4.0,https://api.spotify.com/v1/tracks/1Vo802A38tPF...,track_2,2.0,spotify:track:1Vo802A38tPFHmje1h91um,0.0906,track


## 6. Missing Values

In this section, we check for missing values in each table.  
This helps identify incomplete records and potential issues before further analysis or dataset merging.

In [7]:
for table in table_names:
    
    print(f"\n--- Missing Values: {table} ---")
    
    df = con.execute(f"SELECT * FROM {table}").df()
    
    missing = (
        df.isnull()
        .sum()
        .sort_values(ascending=False)
        .reset_index()
    )
    
    missing.columns = ["column", "missing_values"]
    
    missing = missing[missing["missing_values"] > 0]
    
    display(missing)


--- Missing Values: albums ---


,column,missing_values



--- Missing Values: artists ---


,column,missing_values



--- Missing Values: audio_features ---


...


--- Missing Values: lyrics_features ---


,column,missing_values



--- Missing Values: tracks ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column,missing_values


## 7. Duplicate Records

Duplicate records can negatively affect recommendation quality and statistical analysis.  
Here we check whether duplicate rows exist within each table.

In [8]:
for table in table_names:
    
    df = con.execute(f"SELECT * FROM {table}").df()
    
    duplicates = df.duplicated().sum()
    
    print(f"{table}: {duplicates:,} duplicate rows")

albums: 0 duplicate rows
artists: 0 duplicate rows
audio_features: 0 duplicate rows
lyrics_features: 0 duplicate rows
tracks: 0 duplicate rows


## 8. Basic Numerical Statistics

We inspect summary statistics for numerical variables such as popularity, duration, and audio-related features.  
This helps identify unusual values, outliers, and scaling differences between variables.

In [9]:
for table in table_names:
    
    print(f"\n--- Numerical Summary: {table} ---")
    
    df = con.execute(f"SELECT * FROM {table}").df()
    
    numeric_df = df.select_dtypes(include=["number"])
    
    if numeric_df.shape[1] > 0:
        display(numeric_df.describe())


--- Numerical Summary: albums ---


,column00,total_tracks
count,75511.000000,75511.000000
mean,37755.000000,8.235807
std,21798.292425,11.669811
min,0.000000,1.000000
25%,18877.500000,1.000000
50%,37755.000000,5.000000
75%,56632.500000,12.000000
max,75510.000000,977.000000



--- Numerical Summary: artists ---


,column0,artist_popularity,followers
count,56129.000000,56129.000000,5.612900e+04
mean,28064.000000,34.387447,7.796041e+04
std,16203.190967,16.917287,5.942273e+05
min,0.000000,0.000000,0.000000e+00
25%,14032.000000,22.000000,1.820000e+02
50%,28064.000000,34.000000,1.734000e+03
75%,42096.000000,46.000000,1.520300e+04
max,56128.000000,100.000000,4.156169e+07



--- Numerical Summary: audio_features ---


,column000,Chroma_1,Chroma_10,Chroma_11,Chroma_12,Chroma_2,Chroma_3,Chroma_4,Chroma_5,Chroma_6,...,Tonnetz_3,Tonnetz_4,Tonnetz_5,Tonnetz_6,ZCR,entropy_energy,spectral_bandwith,spectral_centroid,spectral_rollOff_max,spectral_rollOff_min
count,101909.000000,101909.000000,101909.000000,101909.000000,101909.000000,101909.000000,101909.000000,101909.000000,101909.000000,101909.000000,...,101909.000000,101909.000000,101909.000000,101909.000000,101909.000000,101909.000000,101909.000000,101909.000000,101909.000000,101909.000000
mean,50954.000000,0.491150,0.493458,0.471575,0.462471,0.481123,0.487489,0.484347,0.496251,0.493217,...,-0.002073,0.005975,0.001143,0.001435,0.058544,-159.721729,2871.869938,2702.728780,5328.771980,184.972059
std,29418.738629,0.117287,0.116477,0.110333,0.111736,0.109536,0.118358,0.112019,0.118717,0.116065,...,0.047891,0.049153,0.009006,0.009612,0.023824,51.032580,672.087737,1219.417150,1811.378503,110.921078
min,0.000000,0.033948,0.011642,0.011232,0.010693,0.033748,0.020676,0.016802,0.008113,0.007412,...,-0.375904,-0.377345,-0.146779,-0.155205,0.002802,-352.520453,320.831680,0.000000,154.607398,0.308331
25%,25477.000000,0.415335,0.418296,0.401015,0.389863,0.409673,0.410272,0.412065,0.418987,0.417883,...,-0.030148,-0.022488,-0.003508,-0.003766,0.042341,-194.161958,2566.454061,1887.678738,4264.315881,124.732579
50%,50954.000000,0.488723,0.492847,0.471640,0.461675,0.479698,0.484818,0.482367,0.493705,0.491719,...,-0.001515,0.005721,0.001087,0.001021,0.057246,-160.963637,2985.705700,2490.469008,5517.824968,162.874015
75%,76431.000000,0.563149,0.566978,0.540971,0.533485,0.549272,0.560465,0.553222,0.569871,0.565780,...,0.026030,0.034651,0.005764,0.006086,0.072449,-127.664956,3315.728368,3252.257693,6548.460402,213.857040
max,101908.000000,1.000000,0.999983,1.000000,0.999863,1.000000,1.000000,0.999945,1.000000,0.999369,...,0.477255,0.376723,0.086532,0.139061,0.379803,-0.003809,6447.512667,11025.000009,16796.556767,6259.870481



--- Numerical Summary: lyrics_features ---


,column0,mean_syllables_word,mean_words_sentence,n_sentences,n_words,sentence_similarity,vocabulary_wealth
count,94954.000000,94954.000000,94954.000000,94954.000000,94954.000000,94954.000000,94954.000000
mean,47476.500000,1.054448,3.590286,42.949649,316.901995,-0.114327,0.302671
std,27411.003068,0.968960,9.632294,45.001476,871.511267,0.396067,0.586538
min,0.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
25%,23738.250000,1.150000,2.570000,22.000000,125.000000,0.008658,0.400000
50%,47476.500000,1.270000,3.290000,41.000000,238.000000,0.035294,0.540000
75%,71214.750000,1.520000,4.000000,59.000000,356.000000,0.066288,0.630000
max,94953.000000,4.000000,819.200000,2519.000000,39111.000000,0.964286,0.780000



--- Numerical Summary: tracks ---


,column00,acousticness,danceability,disc_number,duration_ms,energy,instrumentalness,key,liveness,loudness,mode,popularity,speechiness,tempo,time_signature,track_number,valence
count,101939.000000,101939.000000,101939.000000,101939.000000,1.019390e+05,101939.000000,101939.000000,101939.000000,101939.000000,101939.000000,101939.000000,101939.000000,101939.000000,101939.000000,101939.000000,101939.000000,101939.000000
mean,50969.000000,0.352124,0.586015,1.032166,2.467708e+05,0.586479,0.148776,5.270858,0.197640,-9.462720,0.618154,39.782311,0.128841,118.358527,3.875651,4.608060,0.482813
std,29427.398883,0.334855,0.177724,0.566789,1.904303e+05,0.260170,0.304024,3.577679,0.175391,6.198508,0.485841,16.790769,0.203324,30.224074,0.517008,7.181805,0.261690
min,0.000000,0.000000,0.000000,1.000000,1.155000e+03,0.000000,0.000000,0.000000,0.000000,-60.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
25%,25484.500000,0.040700,0.480000,1.000000,1.840000e+05,0.411000,0.000000,2.000000,0.095600,-11.149000,0.000000,29.000000,0.036400,95.973000,4.000000,1.000000,0.271000
50%,50969.000000,0.238000,0.610000,1.000000,2.168930e+05,0.629000,0.000037,5.000000,0.124000,-7.599000,1.000000,41.000000,0.050600,118.067000,4.000000,2.000000,0.477000
75%,76453.500000,0.645000,0.714000,1.000000,2.610550e+05,0.798000,0.034400,8.000000,0.241000,-5.509000,1.000000,52.000000,0.104000,136.045000,4.000000,6.000000,0.693000
max,101938.000000,0.996000,0.989000,81.000000,5.505831e+06,1.000000,1.000000,11.000000,0.999000,2.719000,1.000000,97.000000,0.969000,244.035000,5.000000,655.000000,0.993000


## 9. Initial Observations

The initial data overview shows that the Spotify datasets are already relatively clean and well-structured for analytical and recommendation tasks.

Key observations include:
- the datasets are separated into multiple related entities such as tracks, artists, albums, audio features, and lyrics features
- no major missing value issues were detected across the inspected tables
- no fully duplicated rows were identified
- the audio feature dataset contains a large number of numerical variables that can later be used for similarity-based recommendation approaches
- several tables contain nested or list-like structures (e.g. genres, artist IDs, available markets), which may require additional preprocessing during feature engineering
- numerical variables such as popularity, tempo, duration, danceability, and valence already show substantial variation, making them useful candidates for exploratory analysis and recommendation modeling

Overall, the datasets appear suitable for building a content-based music recommendation system. The next steps will focus on deeper exploratory analysis of tracks and audio features before combining datasets into a unified recommendation pipeline.

## 10. Closing Database Connection

Finally, we close the DuckDB connection to avoid file locking issues in later notebooks.

In [10]:
con.close()